Problem 25 – Shannon entropy
----------------------------
Given a batch of **probability distributions** `p` whose entries
sum to 1 along a chosen dimension `dim` (default = −1), compute

        H(p) = − Σᵢ pᵢ log pᵢ          (natural log)

Return a tensor with one fewer dimension than the input, the
summation being performed over `dim`.

* Use **no Python for-loops**.
* Add a small ε before log to avoid `log(0)`.

In [3]:
import torch

def entropy(p: torch.Tensor, dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    """Implement me!"""
    assert torch.all(p >= 0) and torch.allclose(p.sum(dim=dim), torch.tensor(1.0)), "invalid input"
    return (-p * torch.log(p + eps)).sum(dim=dim)
    
# ---------- reference + tests ----------
def _ref_entropy(p, dim=-1, eps=1e-8):
    return -(p * (p + eps).log()).sum(dim)

def _self_check():
    p = torch.tensor([[0.5, 0.5],
                      [0.25, 0.75]])
    try:
        H = entropy(p)
    except NotImplementedError:
        print("🔧  Implement entropy and re-run the cell.")
        return
    assert torch.allclose(H, _ref_entropy(p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 26 – Cross-entropy
--------------------------
Given two *matching* probability tensors **q** (the “true” distribution)
and **p** (the “predicted” distribution) of shape `(..., C)`, compute

        CE(q, p) = − Σᵢ qᵢ log pᵢ

along `dim` (default = −1).  Return a tensor of shape equal to
`q` with that dimension removed.

Assume `q` already sums to 1 (it can be soft or one-hot).
No loops; add ε for log-safety.

In [4]:
import torch

def cross_entropy(q: torch.Tensor, p: torch.Tensor,
                  dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    return (-q * torch.log(p + eps)).sum(dim=dim)


def _ref_cross_entropy(q, p, dim=-1, eps=1e-8):
    return -(q * (p + eps).log()).sum(dim)

def _self_check():
    q = torch.tensor([[0., 1., 0.],
                      [0.2, 0.3, 0.5]])
    p = torch.tensor([[0.1, 0.8, 0.1],
                      [0.3, 0.2, 0.5]])
    try:
        ce = cross_entropy(q, p)
    except NotImplementedError:
        print("🔧  Implement cross_entropy.")
        return
    assert torch.allclose(ce, _ref_cross_entropy(q, p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 27 – KL divergence  D_KL(q || p)
----------------------------------------
For two probability tensors **q** (reference) and **p** (model)
of identical shape `(..., C)`, compute

        KL(q‖p) = Σᵢ qᵢ log(qᵢ / pᵢ)

over `dim` (default = −1).  The output’s shape lacks that dimension.

Requirements
------------
* Pure tensor ops, no loops.
* Add ε to both q and p inside the log to avoid division by zero.

In [5]:
import torch

def kl_divergence(q: torch.Tensor, p: torch.Tensor,
                  dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    return (q * (torch.log(q + eps) - torch.log(p + eps))).sum(dim=dim)


def _ref_kl(q, p, dim=-1, eps=1e-8):
    q_safe = q + eps
    p_safe = p + eps
    return (q_safe * (q_safe / p_safe).log()).sum(dim)

def _self_check():
    q = torch.tensor([[0.6, 0.4]])
    p = torch.tensor([[0.5, 0.5]])
    try:
        kl = kl_divergence(q, p)
    except NotImplementedError:
        print("🔧  Implement kl_divergence.")
        return
    assert torch.allclose(kl, _ref_kl(q, p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 28 – Sinusoidal positional encoding
-------------------------------------------
Implement the original Transformer positional embedding

    PE[pos, 2i]     =  sin( pos / 10000^(2i/d) )
    PE[pos, 2i + 1] =  cos( pos / 10000^(2i/d) )

for sequence length `seq_len` and hidden size `d` (d must be even).

Return a tensor of shape **(seq_len, d)** on the requested
`device` / `dtype` – and **use no Python loops**.

In [7]:
import torch
import math
from typing import Optional

def compute_sin_cos(T, D):
    # sin(t * 10000^(-i/(D/2)))
    # cos(t * 10000^(-i/(D/2)))

    H = D//2
    tt = torch.arange(0, T).view(T, 1)
    ii = torch.arange(0, H).view(1, H)
    ff = 10000**(-ii/H)

    sin = torch.sin(tt * ff)
    cos = torch.cos(tt * ff)

    return sin, cos
    
def sinusoidal_positional_encoding(seq_len: int,
                                   d: int,
                                   device: Optional[torch.device] = None,
                                   dtype: torch.dtype = torch.float32
                                   ) -> torch.Tensor:
    """Implement me!"""
    assert d % 2 == 0, "d must be even"
    sin, cos = compute_sin_cos(seq_len, d)
    pe = torch.stack((sin, cos), dim=-1).reshape(seq_len, d)
    return pe


# -----------------------------------------------------------
def _ref_sinusoidal(seq_len, d, device=None, dtype=torch.float32):
    if d % 2:
        raise ValueError("d must be even.")
    pos = torch.arange(seq_len, device=device).unsqueeze(1)          # (T,1)
    i   = torch.arange(0, d, 2, device=device).unsqueeze(0)          # (1,d/2)
    angle = pos / torch.pow(10000.0, i / d)                          # (T,d/2)
    pe = torch.zeros(seq_len, d, device=device, dtype=dtype)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe

def _self_check():
    T, D = 5, 8
    try:
        pe = sinusoidal_positional_encoding(T, D)
    except NotImplementedError:
        print("🔧  Implement sinusoidal_positional_encoding.")
        return
    assert pe.shape == (T, D), "Shape mismatch"
    assert torch.allclose(pe, _ref_sinusoidal(T, D), atol=1e-6), "Wrong values"
    # Sanity: first position should be [0,1,0,1,...]
    first = pe[0]
    assert torch.allclose(first[0::2], torch.zeros(D//2))            # sin(0)=0
    assert torch.allclose(first[1::2], torch.ones (D//2))            # cos(0)=1
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 29 – Rotary positional embedding (RoPE)
-----------------------------------------------
Apply rotary position encodings (Su et al., 2021) to **query** and
**key** tensors of shape (B, T, d) where d is even.

Algorithm (vector form)
~~~~~~~~~~~~~~~~~~~~~~~
Split last dim into even/odd pairs:

    x_even, x_odd = x[..., 0::2], x[..., 1::2]

Define frequency vector:
    θ_i = 10000^{ -2i / d }   for i = 0 … d/2−1

For each position **pos** (0-based):

    sin  = sin(pos * θ)
    cos  = cos(pos * θ)

Rotate each pair:

    x_rot_even =  x_even * cos - x_odd * sin
    x_rot_odd  =  x_even * sin + x_odd * cos

Interleave the two parts back together to shape (B,T,d).

Return **(q_rot, k_rot)**.

Requirements
------------
* Pure tensor ops, **no explicit Python loops**.

In [10]:
import torch
import math
from typing import Tuple

def compute_sin_cos(T, D):
    # sin(t * 10000^(-i/(D/2)))
    # cos(t * 10000^(-i/(D/2)))

    H = D//2
    tt = torch.arange(0, T).view(T, 1)
    ii = torch.arange(0, H).view(1, H)
    ff = 10000**(-ii/H)

    sin = torch.sin(tt * ff)
    cos = torch.cos(tt * ff)

    return sin, cos

def apply_rope(q: torch.Tensor,
               k: torch.Tensor
               ) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Args
    ----
    q, k : (B, T, d)  with even d

    Returns
    -------
    q_rot, k_rot  – same shapes after rotary encoding.
    """
    B, T, D = q.shape
    sin, cos = compute_sin_cos(T, D)

    def rope(x):
        x0 = x[:, :, 0::2]
        x1 = x[:, :, 1::2]

        y0 = x0 * cos - x1 * sin
        y1 = x0 * sin + x1 * cos

        y = torch.stack((y0, y1), dim=-1).reshape(B, T, D)

        return y

    return rope(q), rope(k)

# -----------------------------------------------------------
def _rope_rotate(x):
    B,T,d = x.shape
    half  = d // 2
    freqs = torch.pow(10000.0, -torch.arange(0, half, device=x.device) / half)
    pos   = torch.arange(T, device=x.device).float()                # (T,)
    sin   = torch.sin( pos.unsqueeze(1) * freqs )                   # (T,half)
    cos   = torch.cos( pos.unsqueeze(1) * freqs )
    sin, cos = sin.unsqueeze(0), cos.unsqueeze(0)                   # (1,T,half)
    x_even, x_odd = x[..., 0::2], x[..., 1::2]                      # (B,T,half)
    rot_even = x_even * cos - x_odd * sin
    rot_odd  = x_even * sin + x_odd * cos
    # interleave
    x_rot = torch.empty_like(x)
    x_rot[..., 0::2] = rot_even
    x_rot[..., 1::2] = rot_odd
    return x_rot

def _ref_apply_rope(q, k):
    return _rope_rotate(q), _rope_rotate(k)

def _self_check():
    q = torch.randn(2, 7, 12)
    k = torch.randn(2, 7, 12)
    try:
        q_r, k_r = apply_rope(q, k)
    except NotImplementedError:
        print("🔧  Implement apply_rope.")
        return
    q_ref, k_ref = _ref_apply_rope(q, k)
    assert torch.allclose(q_r, q_ref, atol=1e-6)
    assert torch.allclose(k_r, k_ref, atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 30 – Connectionist-Temporal Classification (CTC) Loss
------------------------------------------------------------
Write CTC loss **from scratch** (forward + backward) using PyTorch’s
low-level `torch.autograd.Function`.

•  **Signature** must match `torch.nn.CTCLoss`:
       loss = CustomCTCLoss(blank=0, reduction="mean", zero_infinity=False)
       loss(log_probs, targets, input_lengths, target_lengths)

•  **log_probs**  (T, N, C) — log-softmax probabilities  
   **targets**    (N, S)   — padded with a special value (default 0)  
   **input_lengths**  (N,) — actual lengths ≤ T  
   **target_lengths** (N,) — ≤ S

•  Implement the *exclusive* β-variable version of the backward dynamic
   program (β_t(s) excludes the current label at time t).

•  No need to handle GPU / AMP edge cases; loops are fine for the small
  unit tests below, but try to keep your code vectorised where you can.

-----------------------------------------------------------------
Skeleton
~~~~~~~~
1.  **forward**
      – compute CTC loss for each batch element  
      – save anything you need on ctx for backward  
      – return either mean or sum according to `reduction`

2.  **backward**
      – input grad is `grad_out * dLoss/dLogProbs` (shape = log_probs)  
      – return gradients in the order of forward’s inputs  
      – all grads for integer tensors must be **None**

Tips
~~~~
* Expand each target with blanks: e.g. target=[g,a,t] → [blk,g,blk,a,blk,t,blk].
* α/β recurrences (log-space) are stable:
      α_t(s) = logsumexp( α_{t-1}(s)        + lp_t(s),
                           α_{t-1}(s-1)      + lp_t(s),
                           α_{t-1}(s-2) (if label changes) + lp_t(s) )
      β_t(s) = logsumexp( β_{t+1}(s)        + lp_{t+1}(s),
                           β_{t+1}(s+1)      + lp_{t+1}(s+1),
                           β_{t+1}(s+2) (if label changes) + lp_{t+1}(s+2) )
  with **exclusive** β meaning β_t does *not* include lp_t.
* The gradient wrt log_probs is  exp(α+β-logZ), reduced over s.

Feel free to peek at `torch.nn.functional.ctc_loss` for reference, but
do **not** call it inside your implementation.

------------------------------------------------------------
Starter code below — fill in the two `raise NotImplementedError` blocks.

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

NUM_INF = torch.tensor(1e20)

def extend_target(target, blank=0):
    L = len(target)
    out = torch.full((2*L+1,), blank)
    out[1::2] = target
    return out

def logsumexp(a, b):
    a, b = torch.maximum(a, b), torch.minimum(a, b)
    return a + torch.log1p(torch.exp(b-a))

# -------------------  YOUR IMPLEMENTATION AREA  -------------------
class CustomCTCLossFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                log_probs: torch.Tensor,        # (maxT,N,C)  log-softmax already
                targets: torch.Tensor,          # (N,maxL)
                input_lengths: torch.Tensor,    # (N,)
                target_lengths: torch.Tensor,   # (N,)
                blank: int,
                reduction: str,
                zero_infinity: bool,
                gradient_version: str) -> torch.Tensor:
        """
        Returns a scalar (if reduction != 'none') or shape (N,) tensor.
        Save everything you need on ctx for backward.
        """
        maxT, N, C = log_probs.shape
        N, maxL = targets.shape

        losses = []
        grads = torch.zeros_like(log_probs)
        for n in range(N):
            T = input_lengths[n]
            L = target_lengths[n]
            S = 2*L + 1

            lgp = log_probs[:T, n, :]             # T, C
            tgt = targets[n, :L]                  # L
            ext = extend_target(tgt, blank=blank) # 2L+1

            # compute alpha
            alpha = torch.full((T, S), -NUM_INF, dtype=log_probs.dtype)
            alpha[0, 0] = lgp[0, ext[0]]
            alpha[0, 1] = lgp[0, ext[1]]
            #
            for t in range(1, T):
                for s in range(S):
                    # frame t, emission ext[s]
                    stay = alpha[t-1, s-0] + lgp[t, ext[s]]
                    add1 = alpha[t-1, s-1] + lgp[t, ext[s]] if (s-1 >= 0) else -NUM_INF
                    add2 = alpha[t-1, s-2] + lgp[t, ext[s]] if (s-2 >= 0 and ext[s-1] == blank and ext[s-2] != ext[s]) else -NUM_INF

                    alpha[t, s] = logsumexp(logsumexp(stay, add1), add2)

            logZa = logsumexp(alpha[T-1, S-2], alpha[T-1, S-1])

            # compute beta
            beta = torch.full((T, S), -NUM_INF, dtype=log_probs.dtype)
            beta[T-1, S-2] = 0
            beta[T-1, S-1] = 0
            #
            for t in reversed(range(T-1)):
                for s in reversed(range(S)):
                    # frame t, multiple emission possibility
                    stay = lgp[t+1, ext[s+0]] + beta[t+1, s+0]
                    add1 = lgp[t+1, ext[s+1]] + beta[t+1, s+1] if (s+1 < S) else -NUM_INF
                    add2 = lgp[t+1, ext[s+2]] + beta[t+1, s+2] if (s+2 < S and ext[s+1] == blank and ext[s+2] != ext[s]) else -NUM_INF

                    beta[t, s] = logsumexp(logsumexp(stay, add1), add2)

            logZb = logsumexp(lgp[0, ext[0]] + beta[0, 0], lgp[0, ext[1]] + beta[0, 1])

            # compute loss
            assert torch.allclose(logZa, logZb), f"mismatch from alpha and beta computation, {logZa}, {logZb}"
            logZ = (logZa + logZb) / 2
            losses.append(-logZ)

            # compute gamma
            gamma = alpha + beta - logZ

            for s in range(S):
                grads[:T, n, ext[s]] -= torch.exp(gamma[:, s])

            if gradient_version == "torch":
                grads[:T, n, :] += torch.exp(lgp[:T, :])

        losses = torch.tensor(losses)

        ctx.save_for_backward(grads)

        return losses

    
    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        grad_out is either shape (N,) or scalar depending on reduction.
        Must return gradients for the *same* number of inputs received
        in forward:  log_probs, targets, input_lengths, target_lengths,
        blank, reduction, zero_infinity.  Gradients for non-tensor or
        int/str args should be None.
        """

        N, = grad_out.shape
        grads, = ctx.saved_tensors
        grad_in = grad_out.reshape(1, N, 1) * grads
        
        return grad_in, None, None, None, None, None, None, None


class CustomCTCLoss(nn.Module):
    """
    Thin wrapper to match nn.CTCLoss exact call signature.
    """
    def __init__(self,
                 blank: int = 0,
                 reduction: str = "mean",
                 zero_infinity: bool = False,
                 gradient_version: str = "true"):
        super().__init__()
        if reduction not in ("none", "mean", "sum"):
            raise ValueError("reduction must be 'none', 'mean' or 'sum'")
        if gradient_version not in ("true", "torch"):
            raise ValueError("gradient_version must be 'true' or 'torch'")
        self.blank = blank
        self.reduction = reduction
        self.zero_infinity = zero_infinity
        self.gradient_version = gradient_version

    def forward(self,
                log_probs: torch.Tensor,
                targets: torch.Tensor,
                input_lengths: torch.Tensor,
                target_lengths: torch.Tensor) -> torch.Tensor:
        losses = CustomCTCLossFunction.apply(
            log_probs, targets, input_lengths, target_lengths,
            self.blank, self.reduction, self.zero_infinity, self.gradient_version
        )
        if self.reduction == "none":
            loss = losses
        elif self.reduction == "mean":
            loss = torch.mean(losses / target_lengths)
        elif self.reduction == "sum":
            loss = torch.sum(losses)
        return loss
# ------------------------------------------------------------------


# -------------------  REFERENCE + SELF-CHECK  ---------------------
def _ref_ctc_loss(log_probs, targets, input_lengths, target_lengths,
                  blank=0, reduction="mean", zero_infinity=False):
    return F.ctc_loss(log_probs, targets, input_lengths, target_lengths,
                      blank=blank, reduction=reduction,
                      zero_infinity=zero_infinity)

def _self_check():
    torch.manual_seed(0)
    T, N, C = 20, 3, 4
    S = 9
    log_probs = F.log_softmax(torch.randn(T, N, C), dim=-1).requires_grad_()
    targets = torch.randint(1, C, (N, S), dtype=torch.long)
    input_lengths  = torch.full((N,), T, dtype=torch.long)
    target_lengths = torch.randint(1, S+1, (N,), dtype=torch.long)

    my_ctc  = CustomCTCLoss(blank=0, reduction="mean", gradient_version="torch")
    ref_ctc = nn.CTCLoss(blank=0, reduction="mean")

    try:
        loss_my  = my_ctc(log_probs, targets, input_lengths, target_lengths)
    except NotImplementedError:
        print("🔧  Implement CustomCTCLossFunction.forward/backward and re-run.")
        return

    loss_ref = ref_ctc(log_probs, targets, input_lengths, target_lengths)

    # Forward value check
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-4), \
        f"Loss mismatch: {loss_my} vs {loss_ref}"

    # Gradient check (wrt log_probs only)
    loss_ref.backward()
    grad_ref = log_probs.grad.clone()
    log_probs.grad.zero_()
    loss_my.backward()
    grad_my = log_probs.grad

    assert torch.allclose(grad_my, grad_ref, atol=1e-4), \
        "Gradient mismatch w.r.t. log_probs"

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


### Note PyTorch implementation of the CTC gradient is incorrect.

In [45]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
T, N, C = 20, 3, 4
S = 9

log_probs = F.log_softmax(torch.randn(T, N, C, dtype=torch.float64), dim=2).requires_grad_()
targets = torch.randint(1, C, (N, S), dtype=torch.long)
input_lengths = torch.full((N,), T, dtype=torch.long)
target_lengths = torch.randint(1, S + 1, (N,), dtype=torch.long)

custom_ctc_loss = CustomCTCLoss(blank=0, reduction="mean", gradient_version="true")

torch.autograd.gradcheck(custom_ctc_loss, (logits, targets, input_lengths, target_lengths), eps=1e-6, atol=1e-4)

True